In [0]:
from pyspark.sql import functions as F
from datetime import datetime

results = []

def check(name, passed, details=""):
    results.append((datetime.utcnow(), name, "PASS" if passed else "FAIL", str(details)))
    print(f"[{'PASS' if passed else 'FAIL'}] {name} — {details}")

bronze_n = spark.table("fleetpulse.bronze.telemetry_raw").count()
clean_n  = spark.table("fleetpulse.silver.telemetry_clean").count()
quar_n   = spark.table("fleetpulse.silver.telemetry_quarantine").count()
removed  = bronze_n - clean_n - quar_n

# 1. Accounting: nothing lost, nothing invented (Incident 2's legacy)
check("row_accounting", removed >= 0,
      f"bronze={bronze_n}, clean={clean_n}, quarantine={quar_n}, removed_as_dupes={removed}")

# 2. Duplicate ratio in expected band around the generator's 3% (2%–4%)
dup_ratio = removed / bronze_n
check("duplicate_ratio_sane", 0.02 <= dup_ratio <= 0.04, f"{dup_ratio:.4f}")

# 3. Zero duplicates on the true key
dupes = (spark.table("fleetpulse.silver.telemetry_clean")
         .groupBy("event_id").count().filter("count > 1").count())
check("no_dupes_on_event_id", dupes == 0, f"{dupes} duplicate event_ids")

# 4. Clean-table contract: no null keys, no null timestamps
nulls = (spark.table("fleetpulse.silver.telemetry_clean")
         .filter("device_id IS NULL OR event_ts IS NULL").count())
check("no_null_keys_in_clean", nulls == 0, f"{nulls} rows with null keys")

# 5. Quarantine reasons are all known values
valid = {"null_device_id", "unparseable_timestamp", "temp_out_of_range"}
found = {r.reject_reason for r in
         spark.table("fleetpulse.silver.telemetry_quarantine")
              .select("reject_reason").distinct().collect()}
check("quarantine_reasons_known", found <= valid, f"found={found}")

# 6. Referential integrity: every clean device exists in the dimension
orphans = (spark.table("fleetpulse.silver.telemetry_clean").select("device_id")
           .join(spark.table("fleetpulse.silver.devices"), "device_id", "left_anti").count())
check("no_orphan_devices", orphans == 0, f"{orphans} orphan device_ids")

# 7. Alert base-rate guard (Incident 3's legacy): alerts must be rare, never fleet-wide
alert_n = spark.table("fleetpulse.gold.device_alerts").select("device_id").distinct().count()
check("alert_rate_sane", 1 <= alert_n <= 50, f"{alert_n} devices alerting (fleet=500)")

# ---- persist results, then fail loudly if anything failed ----
(spark.createDataFrame(results, "run_ts timestamp, check_name string, status string, details string")
 .write.mode("append").saveAsTable("fleetpulse.ops.dq_results"))

failures = [r for r in results if r[2] == "FAIL"]
if failures:
    raise Exception(f"{len(failures)} DQ check(s) FAILED: {[f[1] for f in failures]}")
print(f"\nAll {len(results)} checks passed.")